# CVライブラリ：日本語OCRのColab動作確認

このノートブックは、PaddleOCRの日本語OCRをCPUで試す確認用です。**別の新しいColabランタイムを開き、ランタイムのタイプをCPUにして、上から順に実行**してください。DETR・SAM用ノートブックと同じランタイムでは実行しません。

Notebook内で英数字のサンプル文字画像を生成します。これは実環境でAPIを試すスモークテストであり、OCRの精度評価ではありません。PaddlePaddleとOCRモデルの取得に通信が発生します。

In [ ]:
import sys
print("Python:", sys.version)
if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError("このライブラリの対応Pythonは3.11〜3.13です。Colabのランタイムを確認してください。")

## 1. CPU版PaddlePaddleを導入

PaddleOCRはPyTorch系の別ランタイムで動作確認します。このノートブックではCPU版を使い、GPU用PaddlePaddleは入れません。

In [ ]:
%pip install "paddlepaddle==3.2.0" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

## 2. 固定した開発版とOCR依存をインストール

コードは次の完全なGit commitから取得します。

In [ ]:
%pip install 'cv-data-lesson[ocr] @ git+https://github.com/708san/CV_data.git@9da8f480bf6bdc9c834a5fe92f371939ad45214b'

In [ ]:
from importlib.metadata import version
from cvlesson import ocr

print("PaddlePaddle:", version("paddlepaddle"))
print("cvlesson devices:", ocr.available_devices())
assert ocr.available_devices()["cpu"] is True

## 3. テスト用の文字画像を作る

Pillowで「TOTAL 1280 JPY」を描きます。レシート画像や個人情報は使いません。

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

image = Image.new("RGB", (1200, 260), "white")
draw = ImageDraw.Draw(image)
font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = ImageFont.truetype(font_path, 92) if Path(font_path).exists() else ImageFont.load_default()
draw.text((40, 65), "TOTAL 1280 JPY", fill="black", font=font)
image_path = "/content/cvlesson_ocr_smoke.png"
image.save(image_path)
display(image)

## 4. OCRを実行

最初の読み込み時に固定版の日本語検出・認識モデルを取得します。結果を出し、文字列・信頼度・位置の形式を確認します。正確な文字列かどうかは、この動作確認では採点しません。

In [ ]:
reader = ocr.load(device="cpu")
result = reader.predict(image_path)
print("読み取り結果:")
print(result.text)
print("行数:", len(result.lines))
assert isinstance(result.text, str)
for line in result.lines:
    assert isinstance(line.text, str)
    assert 0.0 <= line.confidence <= 1.0
    print(line.text, round(line.confidence, 3), line.polygon)
print("OCR推論と戻り値の確認: OK")

## 結果の記録

- Python / PaddlePaddleの版：上の出力
- CPU推論が最後まで完了したか
- 読み取り結果と行数
- エラーが出た場合：最初に失敗したセルと全文を開発者へ共有

この結果はOCR APIが推論を実行できることの確認です。きれいな合成文字での結果を、レシートや手書き文字の精度保証に使わないでください。